In [ ]:
from siluxApi.phntwk.core import StageStructure, NtwkStructure, PhntwkTree
import numpy as np
import pandas as pd

def define_microring_structure():

    coupler = NtwkStructure(
        name="dc_ntwk",
        type="from_stage",
        components=[
            (2, 2, "dc")
        ],
        inherit_ports=(
            {"in":0, "from_ring": 1}, 
            {"out":0, "to_ring": 1}
        )
    )

    ring = NtwkStructure(
        name="ring_ntwk",
        type="from_stage",
        components=[
            StageStructure(
                name="ring_wg",
                type="cascade",
                components=[
                    (1, 1, "bend"),
                    (1, 1, "dut"),
                ],
            )
        ],
        inherit_ports=(
            {"in": 0}, {"out": 0}
        )
    )

    return StageStructure(
        name="microring_tk",
        type="from_ntwk",
        components=[
            NtwkStructure(
                name="all-pass-microring",
                type="ntwk",
                components=[
                    coupler.rename("coupler"),
                    ring.rename("ring")
                ],
                links = [
                    (("coupler", "to_ring"), ("ring", "in")),
                    (("ring", "out"), ("coupler", "from_ring"))
                ]
            )
        ],
        inherit_ports=([("coupler", "in")], [("coupler", "out")])
    )

wavelength = np.arange(1.2, 1.3, 1e-5)

def get_helper():
    structure = define_microring_structure()
    tk = structure.build(wavelength)
    helper = PhntwkTree(tk)
    return tk, helper

In [ ]:

from siluxApi.phntwk.passive import dc
from siluxApi.phntwk.core.transfer_matrix import stage
from siluxApi.phntwk.passive import waveguide as wg


def get_dc(wavelength: np.ndarray, couple_length: float):
    kappa = 0.05 
    theta = 0.41
    df = pd.DataFrame({"wavelength": wavelength})
    df["couple_length"] = couple_length
    df["kappa"] = kappa
    df["theta"] = theta
    df["couple_loss"] = 0.
    df["fanout_loss"] = 0.
    df = dc.props_2_trans_matrix(dc.params_2_props(df))
    return stage.to_matrix(df)

def get_waveguide(wavelength, length):
    neff = 1.65
    df = pd.DataFrame({
        "wavelength": wavelength,
    })
    df["neff"] = neff
    df["length"] = length
    df["loss"] = -1e-4
    df = wg.props_2_trans_matrix(wg.params_2_props(df))
    return stage.to_matrix(df)

def get_dut(wavelength, loss):
    neff = 1.65
    length = 100
    df = pd.DataFrame({
        "wavelength": wavelength,
    })
    df["neff"] = neff
    df["length"] = length
    df["loss"] = loss / length
    df = wg.props_2_trans_matrix(wg.params_2_props(df))
    return stage.to_matrix(df)

In [ ]:
tk, helper = get_helper()
helper.print_tree()

In [ ]:
helper.set_node_matrix("dc", 0, get_dc(wavelength, couple_length=1))
helper.set_node_matrix("bend", 0, get_waveguide(wavelength, length=100))
helper.set_node_matrix("dut", 0, get_dut(wavelength, loss=-3))
helper.get_node_matrix("microring_tk")

In [ ]:
import matplotlib.pyplot as plt

df = stage.to_dataframe(tk)

plt.plot(df["wavelength"], np.abs(df["t_1_1"])**2, label="through_transmission")
plt.legend()

In [ ]:

plt.plot(df["wavelength"], np.log10(np.abs(df["t_1_1"])**2)*10, label="through_transmission_db")
plt.legend()